# Inverse FX Chain Predictor — Gradio test app (effect presence + ORDER)

Pick a **test** pair (dry + wet audio from songs the model never saw), run the model, and compare the predicted
**effect chain** with the **actual** chain underneath. A second tab lets you upload your own dry/wet files.

The model predicts one of **12 audible chains** — EQ and Reverb commute, so e.g. *EQ → Reverb → Compressor* and
*Reverb → EQ → Compressor* sound the same and are both shown as **{EQ, Reverb} → Compressor**.

**Kaggle settings:** Internet **ON** · Add-ons → Secrets → `HF_TOKEN` (read access is enough) · Accelerator: *None* is fine.

**Run:** *Run All*, then open the **public gradio.live link** printed in Section 8 (Kaggle cannot show the app inline).
Run Section 9 to stop it.

## 1. Setup

In [1]:
import os, sys, json, time, shutil, importlib, subprocess
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

if importlib.util.find_spec("gradio") is None or int(importlib.import_module("gradio").__version__.split(".")[0]) < 5:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio>=5.9"])
for mod, spec in {"datasets": "datasets>=2.19", "soundfile": "soundfile", "librosa": "librosa"}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import numpy as np, pandas as pd, torch, soundfile as sf, librosa, gradio as gr
from matplotlib.figure import Figure
print(f"torch {torch.__version__} | gradio {gr.__version__} | Python {sys.version.split()[0]}")

torch 2.11.0+cu128 | gradio 6.26.0 | Python 3.13.15


## 2. Configuration

In [2]:
MODEL_REPO = "APINAJA/inverse-fx-order-logmel-transformer"
DATASET_ID = "APINAJA/inverse-fx-phase1-effect-order"
DATASET_REVISION = "a5d0eeff7f8d9062b30f1971a2bed342e0de2521"   # same dataset version the model was trained on
NUM_DEMO_PAIRS = 100             # spread as evenly as possible over the 12 audible chains (8-9 each)
MAX_PAIRS_PER_SONG = 6           # 22 test songs x 6 >= 100 -> examples come from many songs
MAX_ROWS_TO_SCAN = 990           # whole test split at most (streamed, never fully downloaded)
REBUILD_DEMO = False             # True = pick the demo pairs again

WORK = "/kaggle/working/fx_order_demo" if os.path.exists("/kaggle/working") else os.path.abspath("fx_order_demo")
MODEL_DIR, DEMO_DIR = os.path.join(WORK, "model_repo"), os.path.join(WORK, "demo_data")
os.makedirs(DEMO_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| work dir:", WORK)

device: cuda | work dir: /kaggle/working/fx_order_demo


## 3. Hugging Face login
Token from the Kaggle secret `HF_TOKEN` (never printed).

In [3]:
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
_tok = _tok or os.environ.get("HF_TOKEN")
if _tok:
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import HfApi
    print("Logged in to Hugging Face as:", HfApi(token=_tok).whoami().get("name"))
else:
    print("WARNING: no HF_TOKEN — a private model repo cannot be downloaded.")
del _tok

Logged in to Hugging Face as: APINAJA


## 4. Load the trained model
Downloads `final/` (weights + config), `fxp2/` (the exact model code used in training) and `results/test_predictions.csv` (Kaggle test predictions, used to verify this app).

In [4]:
from huggingface_hub import snapshot_download
snapshot_download(MODEL_REPO, repo_type="model", local_dir=MODEL_DIR, token=os.environ.get("HF_TOKEN"),
                  allow_patterns=["fxp2/*", "final/*", "results/test_predictions.csv"])
for m in [m for m in sys.modules if m == "fxp2" or m.startswith("fxp2.")]:
    del sys.modules[m]                       # re-runs pick up the downloaded code
sys.path.insert(0, MODEL_DIR)

from fxp2.model import build_model, count_parameters
from fxp2.audio import AudioPreprocessor
from fxp2.metrics import decode_chains
from fxp2.common import (EFFECTS, ORDER_QUESTIONS, ORDER_KEYS, CHAINS12, CHAIN12_NAMES, CHAINS15,
                         targets_from_positions)

CFG = json.load(open(os.path.join(MODEL_DIR, "final", "config.json")))["experiment_config"]
model = build_model(CFG)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "final", "model.pt"), map_location="cpu"))
model.to(DEVICE).eval()
PRE = AudioPreprocessor(CFG)
THRESHOLD = float(CFG["THRESHOLD"])
SR, N_SAMPLES = PRE.sample_rate, PRE.num_samples
POS_COLS = CFG["POSITION_COLUMNS"]

state = json.load(open(os.path.join(MODEL_DIR, "final", "training_state.json")))
tm = json.load(open(os.path.join(MODEL_DIR, "final", "test_metrics.json")))
print(f"Model loaded: {count_parameters(model):,} params | best epoch {state['best_epoch']} | fusion {CFG['FUSION']}")
print(f"Kaggle test results: 12-chain accuracy {tm['chain12_accuracy']:.3f} | order accuracy {tm['order/accuracy']:.3f} "
      f"(EQ<Comp {tm['eq_before_comp/accuracy']:.3f}, Rev<Comp {tm['rev_before_comp/accuracy']:.3f}) | "
      f"presence macro F1 {tm['presence/macro_f1']:.3f}")

Model loaded: 2,274,693 params | best epoch 27 | fusion concat_diff
Kaggle test results: 12-chain accuracy 0.472 | order accuracy 0.646 (EQ<Comp 0.557, Rev<Comp 0.735) | presence macro F1 0.914


## 5. Inference helpers
Same preprocessing as training (`fxp2.audio.AudioPreprocessor`: mono, 44.1 kHz, first 10 s / zero-pad, float32, no normalisation). Ground-truth metadata is never passed to the model.

In [5]:
@torch.no_grad()
def predict(dry_wave, wet_wave):
    d = torch.from_numpy(np.ascontiguousarray(dry_wave))[None].to(DEVICE)
    w = torch.from_numpy(np.ascontiguousarray(wet_wave))[None].to(DEVICE)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        out = model(d, w)                        # {"presence": [1,3], "order": [1,2]}  (fp16 on GPU = Kaggle eval)
    pres = torch.sigmoid(out["presence"].float())[0].cpu().numpy()
    order = torch.sigmoid(out["order"].float())[0].cpu().numpy()
    dec = decode_chains(pres[None], order[None])  # joint probability of the 12 audible chains -> argmax
    c = int(dec["pred12"][0])
    return {"presence": pres, "order": order, "pred_presence": (pres >= THRESHOLD).astype(int),
            "pred_order": (order >= 0.5).astype(int), "chain12": c, "chain_name": CHAIN12_NAMES[c],
            "chain_prob": float(dec["chain12_probs"][0, c]), "chain_probs": dec["chain12_probs"][0]}


def load_file_for_model(path):
    """Decode exactly like training (soundfile) when possible; librosa fallback for mp3 etc."""
    try:
        return PRE({"path": path, "bytes": None})
    except Exception:
        arr, sr = librosa.load(path, sr=None, mono=False)
        return PRE({"array": np.atleast_2d(arr).astype(np.float32), "sampling_rate": sr})


def spectrogram_figure(dry_wave, wet_wave, title=""):
    with torch.no_grad():
        lm = model.logmel.raw_log_mel(torch.from_numpy(np.stack([dry_wave, wet_wave])).to(DEVICE)).cpu().numpy()
    diff = lm[1] - lm[0]
    fig = Figure(figsize=(15, 3.3), dpi=90)
    axes = fig.subplots(1, 3, sharey=True)
    lo, hi = np.percentile(lm, 5), lm.max()
    ext = [0, CFG["AUDIO_LENGTH_SECONDS"], 0, CFG["N_MELS"]]
    for ax, m, t in zip(axes[:2], lm, ["Dry log-mel", "Wet log-mel"]):
        ax.imshow(m, origin="lower", aspect="auto", cmap="magma", vmin=lo, vmax=hi, extent=ext)
        ax.set_title(t, loc="left", fontsize=10)
    lim = float(np.percentile(np.abs(diff), 99)) or 1.0
    im = axes[2].imshow(diff, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=ext)
    axes[2].set_title("Wet − Dry (red = louder in wet, blue = quieter)", loc="left", fontsize=10)
    fig.colorbar(im, ax=axes[2], fraction=0.04, pad=0.02, label="dB-like")
    for ax in axes:
        ax.set_xlabel("Time (s)", fontsize=9); ax.tick_params(labelsize=8)
    axes[0].set_ylabel("Mel bin", fontsize=9)
    if title:
        fig.suptitle(title, x=0.01, ha="left", fontsize=10)
    fig.tight_layout()
    return fig


def _fmt_params(p):
    if not p:
        return "—"
    try:
        d = json.loads(p) if isinstance(p, str) else p
    except Exception:
        return str(p)
    units = {"frequency": " Hz", "gain": " dB", "threshold": " dB", "attack": " ms", "release": " ms",
             "makeup_gain": " dB"}
    return ", ".join(f"{k} {v:g}{units.get(k, '')}" if isinstance(v, (int, float)) else f"{k} {v}" for k, v in d.items())


def _yn(v, yes="present", no="absent"):
    return yes if int(v) else no


def result_markdown(res, truth=None):
    pc = CHAINS12[res["chain12"]]                 # (name, short, presence, o1, o2, canonical)
    out = ["### Model prediction", f"**Predicted chain: {res['chain_name']}**  (probability {res['chain_prob']:.2f})", "",
           "| Effect | Probability | Predicted |", "|---|---|---|"]
    out += [f"| {e} | {res['presence'][k]:.3f} | {_yn(res['pred_presence'][k])} |" for k, e in enumerate(EFFECTS)]
    out += ["", "| Order question | P(yes) | Order head says | Used in predicted chain? |", "|---|---|---|---|"]
    for j, q in enumerate(ORDER_QUESTIONS):
        used = pc[3 + j] is not None
        out.append(f"| {q}? | {res['order'][j]:.3f} | {_yn(res['pred_order'][j], 'yes', 'no')} | "
                   f"{'yes' if used else 'no (needed only when both effects are present)'} |")
    if truth is None:
        return "\n".join(out)
    t = truth["targets"]
    chain_ok = res["chain12"] == t["class12"]
    tc = CHAINS12[t["class12"]]
    out += ["", "---", "### Actual (ground truth)",
            f"**True chain as rendered: {' → '.join(truth['chain'])}**",
            f"**Audible chain: {tc[0]}**"]
    if tc[2][0] and tc[2][2] and tc[3] == tc[4]:     # EQ and Reverb on the same side of the compressor (or no compressor)
        out.append("_EQ and Reverb are both linear, so their order relative to each other cannot be heard; "
                   "both renders count as this audible chain._")
    out += ["", "| Effect | Actual | Predicted | Result |", "|---|---|---|---|"]
    for k, e in enumerate(EFFECTS):
        ok = int(res["pred_presence"][k]) == int(t["presence"][k])
        out.append(f"| {e} | {_yn(t['presence'][k])} | {_yn(res['pred_presence'][k])} | {'✅ correct' if ok else '❌ wrong'} |")
    out += ["", "| Order question | Actual | Order head says | Result |", "|---|---|---|---|"]
    for j, q in enumerate(ORDER_QUESTIONS):
        if t["order_mask"][j]:
            ok = int(res["pred_order"][j]) == int(t["order"][j])
            out.append(f"| {q}? | {_yn(t['order'][j], 'yes', 'no')} | {_yn(res['pred_order'][j], 'yes', 'no')} | "
                       f"{'✅ correct' if ok else '❌ wrong'} |")
        else:
            out.append(f"| {q}? | — (not both present) | {_yn(res['pred_order'][j], 'yes', 'no')} | — |")
    out += ["", f"**Chain correct: {'✅ yes' if chain_ok else '❌ no'}**", "",
            "**Settings used to create this wet audio** (shown only for analysis — never given to the model):",
            f"- EQ: {_fmt_params(truth.get('eq_params'))}",
            f"- Compressor: {_fmt_params(truth.get('compressor_params'))}",
            f"- Reverb: {_fmt_params(truth.get('reverb_params'))}"]
    if truth.get("low_effect_change") not in (None, "", "[]"):
        out.append(f"- ⚠️ Flagged as a *low* (subtle) effect change in the dataset: {truth['low_effect_change']}")
    out.append(f"\n<sub>{truth['sample_id']} · {truth['source_category']} · {truth['source_song']}</sub>")
    return "\n".join(out)


def probs_for_label(res):
    """Top chains for the gr.Label bar chart."""
    return {CHAIN12_NAMES[i]: float(res["chain_probs"][i]) for i in range(len(CHAIN12_NAMES))}

import html as _html

# Effect identity colours: fixed categorical slots 1-3 (blue, orange, aqua) — marks only, never text.
FX_COLOR = {"EQ": "#2a78d6", "Compressor": "#eb6834", "Reverb": "#1baf7a"}
GOOD, BAD = "#1baf7a", "#e34948"
_T = "var(--body-text-color, #1f1f1f)"
_T2 = "var(--body-text-color-subdued, #6b6b6b)"
_LINE = "var(--border-color-primary, #d9d9d6)"
_SURF = "var(--block-background-fill, transparent)"


def _esc(x):
    return _html.escape(str(x))


def _chip(effect, faded=False):
    c = FX_COLOR[effect]
    op = "0.35" if faded else "1"
    return (f'<span style="display:inline-flex;align-items:center;gap:6px;padding:6px 12px;border-radius:8px;'
            f'border:1.5px solid {c};background:{c}22;color:{_T};font-weight:600;font-size:14px;opacity:{op}">'
            f'<span style="width:9px;height:9px;border-radius:50%;background:{c}"></span>{_esc(effect)}</span>')


def _node(text):
    return (f'<span style="padding:5px 10px;border-radius:999px;border:1px solid {_LINE};color:{_T2};'
            f'font-size:12px;white-space:nowrap">{_esc(text)}</span>')


_ARROW = f'<span style="color:{_T2};font-size:18px;padding:0 2px">→</span>'


def chain_steps(c12):
    """CHAINS12 entry -> list of steps; a step is an effect name or ("EQ", "Reverb") for an unordered group."""
    _, _, pres, o1, o2, canon = c12
    if pres[0] and pres[2] and o1 == o2:                 # EQ and Reverb on the same side -> order not audible
        group = ("EQ", "Reverb")
        if not pres[1]:
            return [group]
        return [group, "Compressor"] if o1 == 1 else ["Compressor", group]
    return list(canon)


def chain_diagram(c12):
    parts = [_node("dry audio")]
    for step in chain_steps(c12):
        parts.append(_ARROW)
        if isinstance(step, tuple):
            parts.append(f'<span style="display:inline-flex;flex-direction:column;gap:4px;padding:6px;border-radius:10px;'
                         f'border:1.5px dashed {_LINE};align-items:center">'
                         f'<span style="display:flex;gap:6px">{_chip(step[0])}{_chip(step[1])}</span>'
                         f'<span style="font-size:11px;color:{_T2}">any order (not audible)</span></span>')
        else:
            parts.append(_chip(step))
    parts += [_ARROW, _node("wet audio")]
    return f'<div style="display:flex;flex-wrap:wrap;align-items:center;gap:6px;margin:6px 0 2px">{"".join(parts)}</div>'


def _bar(p, color, threshold=0.5):
    w = max(0.0, min(1.0, float(p))) * 100
    return (f'<div style="position:relative;height:12px;border-radius:6px;background:{_LINE};min-width:120px">'
            f'<div style="width:{w:.1f}%;height:100%;border-radius:6px;background:{color}"></div>'
            f'<div title="decision threshold {threshold}" style="position:absolute;left:{threshold * 100:.0f}%;top:-3px;'
            f'width:2px;height:18px;background:{_T}"></div></div>')


def _mark(ok):
    if ok is None:
        return f'<span style="color:{_T2}">—</span>'
    c, sym, txt = (GOOD, "✓", "correct") if ok else (BAD, "✗", "wrong")
    return (f'<span style="display:inline-flex;align-items:center;gap:4px;font-weight:600;color:{_T}">'
            f'<span style="display:inline-grid;place-items:center;width:18px;height:18px;border-radius:50%;'
            f'background:{c};color:#fff;font-size:12px">{sym}</span>{txt}</span>')


def _section(title, body, sub=""):
    sub_html = f'<div style="font-size:12px;color:{_T2};margin-top:2px">{sub}</div>' if sub else ""
    return (f'<div style="border:1px solid {_LINE};border-radius:12px;padding:14px 16px;margin:10px 0;background:{_SURF}">'
            f'<div style="font-size:13px;font-weight:700;letter-spacing:.02em;text-transform:uppercase;color:{_T2}">{title}</div>'
            f'{sub_html}{body}</div>')


def result_html(res, truth=None, chains12=None, effects=None, questions=None, threshold=0.5):
    pc = chains12[res["chain12"]]
    blocks = []
    if truth is not None:                                 # verdict banner (status colour + icon + words)
        t = truth["targets"]
        ok = res["chain12"] == t["class12"]
        c = GOOD if ok else BAD
        msg = ("Correct — the model found the actual effect chain." if ok else
               f"Wrong — predicted <b>{_esc(pc[0])}</b>, actual <b>{_esc(chains12[t['class12']][0])}</b>.")
        blocks.append(f'<div style="display:flex;align-items:center;gap:10px;padding:12px 16px;border-radius:12px;'
                      f'border:1.5px solid {c};background:{c}1f;color:{_T};font-size:15px">'
                      f'<span style="display:inline-grid;place-items:center;width:26px;height:26px;border-radius:50%;'
                      f'background:{c};color:#fff;font-weight:700;flex:none">{"✓" if ok else "✗"}</span><span>{msg}</span></div>')

    # chain diagrams
    pred_body = (chain_diagram(pc) +
                 f'<div style="font-size:13px;color:{_T2}">confidence {res["chain_prob"]:.0%} (probability of this chain)</div>')
    blocks.append(_section("Model prediction", pred_body))
    if truth is not None:
        t = truth["targets"]
        tc = chains12[t["class12"]]
        rendered = " → ".join(truth["chain"])
        blocks.append(_section("Ground truth", chain_diagram(tc), f"rendered in the dataset as: {_esc(rendered)}"))

    # presence bars
    rows = []
    for k, e in enumerate(effects):
        p = float(res["presence"][k])
        pred = "present" if res["pred_presence"][k] else "absent"
        actual, ok = "", None
        if truth is not None:
            a = int(truth["targets"]["presence"][k])
            actual = "present" if a else "absent"
            ok = int(res["pred_presence"][k]) == a
        rows.append(f'<tr><td style="padding:6px 8px 6px 0">{_chip(e, faded=not res["pred_presence"][k])}</td>'
                    f'<td style="padding:6px 8px;width:45%">{_bar(p, FX_COLOR[e], threshold)}</td>'
                    f'<td style="padding:6px 8px;color:{_T};font-variant-numeric:tabular-nums">{p:.0%}</td>'
                    f'<td style="padding:6px 8px;color:{_T};white-space:nowrap">model: <b>{pred}</b></td>'
                    + (f'<td style="padding:6px 8px;color:{_T};white-space:nowrap">actual: <b>{actual}</b></td><td style="padding:6px 0">{_mark(ok)}</td>'
                       if truth is not None else "") + "</tr>")
    blocks.append(_section("Which effects are present?", f'<table style="width:100%;border-collapse:collapse;margin-top:6px">{"".join(rows)}</table>',
                           "bar = model probability · tick = 50 % decision threshold"))

    # order questions
    rows = []
    pairs = [("EQ", "Compressor"), ("Reverb", "Compressor")]
    for j, q in enumerate(questions):
        a_eff, b_eff = pairs[j]
        p = float(res["order"][j])
        used = pc[3 + j] is not None
        defined = bool(truth["targets"]["order_mask"][j]) if truth is not None else None
        relevant = used or bool(defined)
        model_ans = "before" if res["pred_order"][j] else "after"
        if truth is not None and defined:
            act = "before" if truth["targets"]["order"][j] else "after"
            ok = int(res["pred_order"][j]) == int(truth["targets"]["order"][j])
        elif truth is not None:
            act, ok = "n/a", None
        else:
            act, ok = "", None
        dim = "" if relevant else "opacity:.45;"
        note = "" if relevant else f'<div style="font-size:11px;color:{_T2}">not needed — both effects are not present</div>'
        rows.append(f'<tr style="{dim}"><td style="padding:8px 8px 8px 0;color:{_T};white-space:nowrap"><b>{_esc(q)}?</b>{note}</td>'
                    f'<td style="padding:8px;width:38%">{_bar(p, FX_COLOR[a_eff])}'
                    f'<div style="display:flex;justify-content:space-between;font-size:11px;color:{_T2};margin-top:3px">'
                    f'<span>after</span><span>before</span></div></td>'
                    f'<td style="padding:8px;color:{_T};font-variant-numeric:tabular-nums">{p:.0%}</td>'
                    f'<td style="padding:8px;color:{_T};white-space:nowrap">model: <b>{_esc(model_ans)}</b></td>'
                    + (f'<td style="padding:8px;color:{_T};white-space:nowrap">actual: <b>{_esc(act)}</b></td><td style="padding:8px 0">{_mark(ok)}</td>'
                       if truth is not None else "") + "</tr>")
    blocks.append(_section("Order — where does the Compressor sit?",
                           f'<table style="width:100%;border-collapse:collapse;margin-top:6px">{"".join(rows)}</table>',
                           "EQ ↔ Reverb order is not audible, so only each effect's position relative to the Compressor is predicted"))
    return f'<div style="font-family:inherit;max-width:1100px">{"".join(blocks)}</div>'


def result_view(res, truth=None):
    return result_html(res, truth, chains12=CHAINS12, effects=EFFECTS, questions=ORDER_QUESTIONS, threshold=THRESHOLD)

print("helpers ready")

helpers ready


## 6. Pick the test pairs
Streams the **test** split (no full download; ~2 GB read once, a few minutes) and keeps `NUM_DEMO_PAIRS` pairs spread evenly over the 12 audible chains, Bass / Drums / Mixture and many songs. Saved to `demo_data/` (~0.5 GB) so re-runs are instant; set `REBUILD_DEMO = True` after changing the number.

In [6]:
CATS = ["Bass", "Drums", "Mixture"]
DEMO_JSON = os.path.join(DEMO_DIR, "demo.json")


def _targets(row):
    return targets_from_positions(tuple(int(row[c]) for c in POS_COLS))


def chain_quota(total, n_cls):
    """Even split: e.g. 100 over 12 chains -> 4 chains x 9 + 8 chains x 8."""
    return {c: total // n_cls + (1 if c < total % n_cls else 0) for c in range(n_cls)}


def select_rows(rows, total=NUM_DEMO_PAIRS, max_per_song=MAX_PAIRS_PER_SONG, max_rows=MAX_ROWS_TO_SCAN):
    """Deterministic: slot s of audible chain c prefers category CATS[(c + s) % 3]; <= max_per_song picks per song."""
    n_cls = len(CHAINS12)
    quota = chain_quota(total, n_cls)
    picked = {c: [] for c in range(n_cls)}
    backup = {c: [] for c in range(n_cls)}
    per_song = {}
    for n, row in enumerate(rows):
        if n >= max_rows:
            break
        c = _targets(row)["class12"]
        slot = len(picked[c])
        if slot < quota[c] and row["source_category"] == CATS[(c + slot) % 3] \
                and per_song.get(row["source_song"], 0) < max_per_song:
            picked[c].append(row)
            per_song[row["source_song"]] = per_song.get(row["source_song"], 0) + 1
        elif len(backup[c]) < quota[c]:
            backup[c].append(row)
        if all(len(picked[c]) == quota[c] for c in range(n_cls)):
            break
    for c in range(n_cls):                           # fallback if the preferred spread was not found
        for row in backup[c]:
            if len(picked[c]) < quota[c] and all(row is not r for r in picked[c]):
                picked[c].append(row)
    # interleave chains so neighbouring pairs in the list are different chains
    out, k = [], 0
    while any(picked.values()):
        c = k % n_cls
        if picked[c]:
            out.append(picked[c].pop(0))
        k += 1
    return out


def _write_pair(i, row):
    base = os.path.join(DEMO_DIR, f"pair{i:02d}")
    files = {}
    for kind in ("dry", "wet"):
        val = row[f"{kind}_audio"]
        ext = os.path.splitext(val.get("path") or "")[1] or ".wav"
        raw = f"{base}_{kind}{ext}"                  # original bytes -> model input (exactly as in training)
        with open(raw, "wb") as f:
            f.write(val["bytes"])
        files[kind] = raw
    d, _ = sf.read(files["dry"], dtype="float32", always_2d=True)
    w, sr = sf.read(files["wet"], dtype="float32", always_2d=True)
    scale = 0.99 / max(0.99, float(np.abs(d).max()), float(np.abs(w).max()))   # same gain for both: keeps level difference
    for kind, arr in (("dry", d), ("wet", w)):
        play = f"{base}_{kind}_play.wav"             # 16-bit copy for the browser player
        sf.write(play, arr.mean(axis=1) * scale, sr, subtype="PCM_16")
        files[f"{kind}_play"] = play
    keep = ["sample_id", "source_category", "source_song", "eq_params", "compressor_params", "reverb_params",
            "effect_change_db", "low_effect_change", *POS_COLS]
    meta = {k: row.get(k) for k in keep}
    t = _targets(row)
    meta.update(files, targets={k: (list(v) if isinstance(v, tuple) else v) for k, v in t.items()},
                chain=list(t["chain"]), chain12_name=CHAIN12_NAMES[t["class12"]],
                pair=i, display=f"Pair {i:02d} · {row['source_category']} · {row['source_song']}")
    return meta


_cached = json.load(open(DEMO_JSON)) if os.path.exists(DEMO_JSON) else None
if _cached is not None and not REBUILD_DEMO and len(_cached) == NUM_DEMO_PAIRS:
    DEMO = _cached
    print(f"Loaded {len(DEMO)} cached demo pairs from {DEMO_JSON}")
else:
    if _cached is not None:
        print(f"Rebuilding the demo set ({len(_cached)} cached pairs, {NUM_DEMO_PAIRS} requested)")
    shutil.rmtree(DEMO_DIR, ignore_errors=True); os.makedirs(DEMO_DIR, exist_ok=True)
    from datasets import load_dataset, Audio
    t0 = time.time()
    stream = load_dataset(DATASET_ID, split="test", streaming=True, revision=DATASET_REVISION,
                          token=os.environ.get("HF_TOKEN"))
    stream = stream.cast_column("dry_audio", Audio(decode=False)).cast_column("wet_audio", Audio(decode=False))
    rows = select_rows(stream)
    DEMO = []
    for i, r in enumerate(rows):
        DEMO.append(_write_pair(i + 1, r))
        if (i + 1) % 25 == 0:
            print(f"  saved {i + 1}/{len(rows)} pairs")
    json.dump(DEMO, open(DEMO_JSON, "w"), indent=1)
    print(f"Selected and saved {len(DEMO)} test pairs in {time.time() - t0:.0f}s")

demo_df = pd.DataFrame([{"pair": d["pair"], "category": d["source_category"], "song": d["source_song"],
                         "true chain (rendered)": " → ".join(d["chain"]), "audible chain": d["chain12_name"]} for d in DEMO])
print(f"{len(DEMO)} pairs | {demo_df['song'].nunique()} songs | per category: {demo_df['category'].value_counts().to_dict()}")
display(demo_df.groupby("audible chain").size().reindex(CHAIN12_NAMES).rename("pairs").to_frame())

  saved 25/100 pairs
  saved 50/100 pairs
  saved 75/100 pairs
  saved 100/100 pairs
Selected and saved 100 test pairs in 61s
100 pairs | 18 songs | per category: {'Mixture': 34, 'Bass': 33, 'Drums': 33}


,pairs
audible chain,
EQ,9
Compressor,9
Reverb,9
EQ + Reverb (order not audible),9
EQ → Compressor,8
Compressor → EQ,8
Compressor → Reverb,8
Reverb → Compressor,8
"Compressor → {EQ, Reverb}",8


## 7. Verify the app against the Kaggle test run
The app's probabilities must match `results/test_predictions.csv` (produced during training). A match proves the preprocessing here is identical to training. Small differences (≈1e-3) are normal on CPU (fp32) vs Kaggle GPU (fp16).

In [7]:
ref = pd.read_csv(os.path.join(MODEL_DIR, "results", "test_predictions.csv")).set_index("sample_id")
RESULTS = {}
rows_v = []
PROB_COLS = [f"p_{e}" for e in EFFECTS] + [f"p_{q}" for q in ORDER_KEYS]
t0 = time.time()
for n_done, d in enumerate(DEMO, 1):
    res = predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    RESULTS[d["pair"]] = res
    if n_done % 25 == 0:
        print(f"  {n_done}/{len(DEMO)} pairs predicted ({time.time() - t0:.0f}s)")
    mine = np.concatenate([res["presence"], res["order"]])
    r = ref.loc[d["sample_id"]] if d["sample_id"] in ref.index else None
    diff = float(np.max(np.abs(mine - r[PROB_COLS].to_numpy(dtype=float)))) if r is not None else np.nan
    flips = int(res["chain_name"] != r["pred_chain12"]) if r is not None else np.nan
    rows_v.append({"pair": d["pair"], "sample_id": d["sample_id"], "max |Δprob|": diff, "decision flips": flips})
ver = pd.DataFrame(rows_v).set_index("pair")
display(ver.sort_values("max |Δprob|", ascending=False).head(10))   # worst 10 shown; all are checked below
max_diff = float(np.nanmax(ver["max |Δprob|"])) if ver["max |Δprob|"].notna().any() else float("nan")
if np.isnan(max_diff):
    print("⚠️ None of the demo pairs were found in test_predictions.csv — cannot verify.")
elif max_diff < 0.02 and int(np.nansum(ver["decision flips"])) == 0:
    print(f"✅ VERIFIED: app matches the Kaggle evaluation (max difference {max_diff:.2e}).")
else:
    print(f"❌ MISMATCH: max difference {max_diff:.3f}. The app is not processing audio like training — do not trust it.")

  25/100 pairs predicted (1s)
  50/100 pairs predicted (1s)
  75/100 pairs predicted (2s)
  100/100 pairs predicted (2s)


,sample_id,max |Δprob|,decision flips
pair,,,
14,9-Mixture-Compressor,0.001635,0
17,9-Mixture-EQ-Compressor,0.000732,0
36,90-Drums-EQ-Reverb-Compressor,0.000545,0
73,32-Bass-EQ,0.000421,0
45,91-Mixture-Compressor-EQ-Reverb,0.000348,0
48,91-Mixture-EQ-Reverb-Compressor,0.000341,0
71,96-Bass-Reverb-Compressor-EQ,0.000267,0
8,4-Drums-Reverb-Compressor,0.000236,0
43,49-Bass-Compressor-Reverb,0.000228,0


✅ VERIFIED: app matches the Kaggle evaluation (max difference 1.63e-03).


## 8. Launch the Gradio app
Open the **`https://….gradio.live`** link printed below.

In [10]:
CHOICES = [(d["display"], d["pair"]) for d in DEMO]
BY_PAIR = {d["pair"]: d for d in DEMO}
import random as _random


def _is_correct(d):
    return RESULTS[d["pair"]]["chain12"] == d["targets"]["class12"]


def filtered_choices(cat, result):
    out = []
    for d in DEMO:
        if cat != "All" and d["source_category"] != cat:
            continue
        if result == "Correct only" and not _is_correct(d):
            continue
        if result == "Wrong only" and _is_correct(d):
            continue
        out.append((d["display"], d["pair"]))
    return out


def on_filter(cat, result):
    ch = filtered_choices(cat, result)
    note = f"{len(ch)} of {len(DEMO)} pairs match"
    return gr.update(choices=ch, value=ch[0][1] if ch else None), note


def on_step(pair, cat, result, step):
    ch = [v for _, v in filtered_choices(cat, result)]
    if not ch:
        return gr.update()
    i = ch.index(int(pair)) if pair is not None and int(pair) in ch else -1
    return gr.update(value=ch[(i + step) % len(ch)])


def on_random(cat, result):
    ch = [v for _, v in filtered_choices(cat, result)]
    return gr.update(value=_random.choice(ch)) if ch else gr.update()


def on_select(pair):
    if pair is None:
        return None, None, None, None, "", ""
    d = BY_PAIR[int(pair)]
    fig = spectrogram_figure(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]), d["display"])
    return (d["dry_play"], d["wet_play"], fig, None,
            "<div style='padding:12px 0;color:var(--body-text-color-subdued)'>Press <b>Run model</b> to see the "
            "predicted chain and the actual chain.</div>", "")


def on_run(pair):
    d = BY_PAIR[int(pair)]
    res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
    return probs_for_label(res), result_view(res, d), result_markdown(res, d)


def on_run_all():
    rows = []
    n_chain = n_pres = n_ord = n_ord_def = 0
    for d in DEMO:
        res = RESULTS.get(d["pair"]) or predict(load_file_for_model(d["dry"]), load_file_for_model(d["wet"]))
        t = d["targets"]
        chain_ok = res["chain12"] == t["class12"]
        pres_ok = all(int(res["pred_presence"][k]) == int(t["presence"][k]) for k in range(3))
        defined = [j for j in range(2) if t["order_mask"][j]]
        ord_hits = [int(res["pred_order"][j]) == int(t["order"][j]) for j in defined]
        n_chain += chain_ok; n_pres += pres_ok; n_ord += sum(ord_hits); n_ord_def += len(defined)
        rows.append({"Pair": d["pair"], "Category": d["source_category"], "True (rendered)": " → ".join(d["chain"]),
                     "True (audible)": d["chain12_name"], "Predicted": res["chain_name"],
                     "p(pred)": round(res["chain_prob"], 3), "Chain": "✅" if chain_ok else "❌",
                     "Presence": "✅" if pres_ok else "❌",
                     "Order": ("✅" if all(ord_hits) else "❌") if defined else "—"})
    n = len(DEMO)
    summary = (f"**{n} test pairs** · chain correct **{n_chain}/{n} ({n_chain / n:.0%})** · presence all-correct "
               f"{n_pres}/{n} · order questions correct {n_ord}/{n_ord_def}")
    df = pd.DataFrame(rows)
    per_chain = (df.assign(ok=df["Chain"] == "✅").groupby("True (audible)")["ok"].agg(["sum", "size"])
                 .reindex(CHAIN12_NAMES).dropna().astype(int).reset_index())
    per_chain.columns = ["True chain", "Correct", "Pairs"]
    per_chain["Accuracy"] = (per_chain["Correct"] / per_chain["Pairs"]).map("{:.0%}".format)
    return df, summary, per_chain


def on_upload(dry_path, wet_path):
    if not dry_path or not wet_path:
        return "Please upload **both** a dry and a wet file.", None, None, "", ""
    notes = []
    for name, p in (("Dry", dry_path), ("Wet", wet_path)):
        try:
            info = sf.info(p); sr, ch, dur = info.samplerate, info.channels, info.duration
        except Exception:
            y, sr = librosa.load(p, sr=None, mono=False); ch = 1 if y.ndim == 1 else y.shape[0]; dur = y.shape[-1] / sr
        if sr != SR: notes.append(f"{name}: {sr} Hz → resampled to {SR} Hz")
        if ch > 1: notes.append(f"{name}: {ch} channels → mixed to mono")
        if dur > CFG["AUDIO_LENGTH_SECONDS"] + 0.01: notes.append(f"{name}: {dur:.1f} s → only the first {CFG['AUDIO_LENGTH_SECONDS']:.0f} s is used")
        if dur < CFG["AUDIO_LENGTH_SECONDS"] - 0.01: notes.append(f"{name}: {dur:.1f} s → zero-padded to {CFG['AUDIO_LENGTH_SECONDS']:.0f} s")
        if name == "Dry": dry_dur = dur
        else: wet_dur = dur
    if abs(dry_dur - wet_dur) > 0.05:
        notes.append(f"Dry ({dry_dur:.2f} s) and wet ({wet_dur:.2f} s) differ in length — they must be the same, time-aligned recording")
    dry, wet = load_file_for_model(dry_path), load_file_for_model(wet_path)
    res = predict(dry, wet)
    warn = ("**Notes:**\n" + "\n".join(f"- {x}" for x in notes) + "\n\n" if notes else "") + \
           "_The model was trained on Pedalboard EQ/Compressor/Reverb applied to 10 s MUSDB-style stems; " \
           "other audio or other plugins may give unreliable results._"
    return warn, spectrogram_figure(dry, wet, "Uploaded pair"), probs_for_label(res), result_view(res), result_markdown(res)


with gr.Blocks(title="Inverse FX Predictor — effect order") as demo:
    gr.Markdown(f"## Inverse FX Predictor — which effects, in which order?\n"
                f"Model `{MODEL_REPO}` (two-head Log-Mel Transformer, best epoch {state['best_epoch']}). "
                f"Inputs are only the dry and wet audio. The output is the most probable of the 12 audible chains "
                f"(EQ↔Reverb order is not audible, so those renders share one chain).")
    with gr.Tabs():
        with gr.Tab("Test pairs"):
            with gr.Row():
                cat_filter = gr.Radio(["All", "Bass", "Drums", "Mixture"], value="All", label="Source type")
                res_filter = gr.Radio(["All", "Correct only", "Wrong only"], value="All", label="Model result")
                filter_note = gr.Markdown(f"{len(DEMO)} of {len(DEMO)} pairs match")
            with gr.Row():
                prev_btn = gr.Button("◀ Prev", scale=0, min_width=90)
                pick = gr.Dropdown(choices=CHOICES, value=CHOICES[0][1], label="Test pair (true answer hidden)", scale=4)
                next_btn = gr.Button("Next ▶", scale=0, min_width=90)
                rand_btn = gr.Button("🎲 Random", scale=0, min_width=110)
                run_btn = gr.Button("Run model", variant="primary", scale=1)
            with gr.Row():
                dry_player = gr.Audio(label="Dry (input)", type="filepath", interactive=False)
                wet_player = gr.Audio(label="Wet (processed)", type="filepath", interactive=False)
            result_card = gr.HTML()
            with gr.Row():
                with gr.Column(scale=1):
                    prob_label = gr.Label(label="Top 5 most probable chains", num_top_classes=5)
                with gr.Column(scale=2):
                    with gr.Accordion("Detailed tables", open=False):
                        result_md = gr.Markdown()
            with gr.Accordion("Log-mel spectrograms (dry, wet, wet − dry)", open=False):
                spec_plot = gr.Plot(label="Log-mel spectrograms")
            with gr.Accordion("Run all demo pairs", open=False):
                all_btn = gr.Button("Run all")
                all_summary = gr.Markdown()
                all_per_chain = gr.Dataframe(interactive=False, label="Accuracy per true chain")
                all_table = gr.Dataframe(interactive=False, wrap=True, label="All pairs", max_height=500)
        with gr.Tab("Upload your own"):
            with gr.Row():
                up_dry = gr.Audio(label="Dry audio", type="filepath")
                up_wet = gr.Audio(label="Wet audio", type="filepath")
            up_btn = gr.Button("Predict effect chain", variant="primary")
            up_notes = gr.Markdown()
            up_card = gr.HTML()
            with gr.Row():
                with gr.Column(scale=1):
                    up_label = gr.Label(label="Top 5 most probable chains", num_top_classes=5)
                with gr.Column(scale=2):
                    with gr.Accordion("Detailed tables", open=False):
                        up_md = gr.Markdown()
            with gr.Accordion("Log-mel spectrograms (dry, wet, wet − dry)", open=False):
                up_plot = gr.Plot(label="Log-mel spectrograms")

    sel_out = [dry_player, wet_player, spec_plot, prob_label, result_card, result_md]
    pick.change(on_select, inputs=pick, outputs=sel_out)
    demo.load(on_select, inputs=pick, outputs=sel_out)
    run_btn.click(on_run, inputs=pick, outputs=[prob_label, result_card, result_md])
    all_btn.click(on_run_all, outputs=[all_table, all_summary, all_per_chain])
    for f in (cat_filter, res_filter):
        f.change(on_filter, inputs=[cat_filter, res_filter], outputs=[pick, filter_note])
    prev_btn.click(lambda p, c, r: on_step(p, c, r, -1), inputs=[pick, cat_filter, res_filter], outputs=pick)
    next_btn.click(lambda p, c, r: on_step(p, c, r, +1), inputs=[pick, cat_filter, res_filter], outputs=pick)
    rand_btn.click(on_random, inputs=[cat_filter, res_filter], outputs=pick)
    up_btn.click(on_upload, inputs=[up_dry, up_wet], outputs=[up_notes, up_plot, up_label, up_card, up_md])

demo.queue(default_concurrency_limit=2).launch(share=True, inline=False, allowed_paths=[WORK], show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2b968adfdac3342c9e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 9. Stop the app

In [9]:
demo.close()
print("Gradio app stopped.")

Closing server running on port: 7860
Gradio app stopped.
